# Initial setup

In [205]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Testing Semantic Similarity using Cosine functions using Gemini Embedding Model

In [206]:
from google import genai
from google.genai import types
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

client = genai.Client()

texts = [
    "What is the meaning of life?",
    "What is the purpose of existence?",
    "How do I bake a cake?"]

result = [
    np.array(e.values) for e in client.models.embed_content(
        model="gemini-embedding-001",
        contents=texts,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")).embeddings
]

# Calculate cosine similarity. Higher scores = greater semantic similarity.

embeddings_matrix = np.array(result)
similarity_matrix = cosine_similarity(embeddings_matrix)

for i, text1 in enumerate(texts):
    for j in range(i + 1, len(texts)):
        text2 = texts[j]
        similarity = similarity_matrix[i, j]
        print(f"Similarity between '{text1}' and '{text2}': {similarity:.4f}")

Similarity between 'What is the meaning of life?' and 'What is the purpose of existence?': 0.9417
Similarity between 'What is the meaning of life?' and 'How do I bake a cake?': 0.7676
Similarity between 'What is the purpose of existence?' and 'How do I bake a cake?': 0.7471


In [207]:
# !pip install langchain-google-genai langchain-chroma chromadb

### Initialize the native Chroma client


In [208]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="/content/drive/MyDrive/content")
local_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2" # Using Local Embedding Model
)


# Loading data into notebook

In [209]:
import csv

ids=[]
cgpa=[]
documents=[]
metadatas=[]

with open('/content/drive/MyDrive/content/career_dataset.csv', mode='r', encoding='latin-1') as csvfile:
  reader=csv.DictReader(csvfile, skipinitialspace=True)

  for i, row in enumerate(reader):
    ids.append(row['id'])

    metadatas.append({
        "education": row['Education Level'],
        "career": row['Recommended Career'],
        "specialization": row['Specialization']
    })

    text = (
        f"Education: {row['Education Level']} in {row['Specialization']}. "
        f"Skills: {row['Skills']}. "
        f"Certifications: {row['Certifications']}. "
        f"Recommended Career Path: {row['Recommended Career']}."
    )
    documents.append(text)


In [210]:
print(cgpa[:1])
print(metadatas[:1])
print(documents[:1])

[]
[{'education': "Bachelor's", 'career': 'Business Analyst', 'specialization': 'Finance'}]
["Education: Bachelor's in Finance. Skills: Counseling, MS Office, Machine Learning. Certifications: Tally ERP. Recommended Career Path: Business Analyst."]


## Create job collection

In [211]:
job_collection = client.get_or_create_collection(
    name="jobs",
    embedding_function=local_ef
)

## Data ingestion

In [212]:
job_collection.add(ids=ids, metadatas=metadatas, documents=documents)
print(f"Successfully added {job_collection.count()} items to the job_collection.")

Successfully added 99 items to the job_collection.


### Semantic Search application

In [213]:
education_value="Bachelor's" # Choose from 'Bachelor's', 'Master's', 'PhD', 'Matric', 'Intermediate'
specialization_value="Science"
skills_value="Python MS Office"

query_strings = [f"I have a {education_value} in {specialization_value} and know {skills_value}. What career is best?"]

In [214]:
result=job_collection.query(
    query_texts=query_strings,
    n_results=5
    )

print(result)

{'ids': [['90', '89', '53', '2', '7']], 'embeddings': None, 'documents': [["Education: Master's in Arts. Skills: Accounting, Python. Certifications: Creative Writing. Recommended Career Path: Clerk.", 'Education: Intermediate in Computer Science. Skills: Counseling, SQL, Python. Certifications: AWS Certified. Recommended Career Path: Clerk.', 'Education: PhD in Arts. Skills: Communication, Python, Accounting. Certifications: Creative Writing. Recommended Career Path: Data Entry Operator.', 'Education: Intermediate in Science. Skills: Accounting, MS Office. Certifications: AWS Certified. Recommended Career Path: Software Engineer.', 'Education: Matric in Computer Science. Skills: MS Office, Financial Analysis. Certifications: Mental Health Basics. Recommended Career Path: Clerk.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'education': "Master's", 'specialization': 'Arts', 'career': 'Clerk'}, {'career': 'Clerk', 'education': 'Inter

### Structured Output response

In [215]:
for doc, meta, distance in zip(
    result['documents'][0],
    result['metadatas'][0],
    result['distances'][0]
):

    # distance of 0.0 is a 100% match
    match_score = max(0, 100 - round(distance * 100, 2))

    print(f"Recommended Career: {meta['career']}")
    print(f"Match Confidence: {match_score}%")
    print(f"Background: {meta['education']}")
    print(f"Profile Summary: {doc}")
    print(f"Distance Metric: {round(distance, 4)}")
    print("-" * 50)

Recommended Career: Clerk
Match Confidence: 64.58%
Background: Master's
Profile Summary: Education: Master's in Arts. Skills: Accounting, Python. Certifications: Creative Writing. Recommended Career Path: Clerk.
Distance Metric: 0.3542
--------------------------------------------------
Recommended Career: Clerk
Match Confidence: 61.38%
Background: Intermediate
Profile Summary: Education: Intermediate in Computer Science. Skills: Counseling, SQL, Python. Certifications: AWS Certified. Recommended Career Path: Clerk.
Distance Metric: 0.3862
--------------------------------------------------
Recommended Career: Data Entry Operator
Match Confidence: 61.07%
Background: PhD
Profile Summary: Education: PhD in Arts. Skills: Communication, Python, Accounting. Certifications: Creative Writing. Recommended Career Path: Data Entry Operator.
Distance Metric: 0.3893
--------------------------------------------------
Recommended Career: Software Engineer
Match Confidence: 60.46%
Background: Intermedi